In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
# Load the cleaned customer dataset
clients = pd.read_csv("clients_cleaned.csv")

print("Dataset shape:", clients.shape)
print("\nColumns:")
print(clients.columns.tolist())

display(clients.head())


Dataset shape: (2000, 12)

Columns:
['client_id', 'client_type', 'first_name', 'last_name', 'date_of_birth', 'gender', 'country', 'region', 'acquisition_purpose', 'satisfaction_score', 'loan_applied', 'referral_channel']


,client_id,client_type,first_name,last_name,date_of_birth,gender,country,region,acquisition_purpose,satisfaction_score,loan_applied,referral_channel
0,C0001,Individual,Kareem,Liu,1968-05-11,F,USA,California,Home,4,Yes,Website
1,C0002,Individual,Trystan,Oconnor,1962-11-26,M,USA,California,Home,1,No,Website
2,C0003,Individual,Kale,Gay,1959-04-07,M,USA,California,Home,4,Yes,Agency
3,C0004,Individual,Russell,Gross,1959-11-25,M,USA,California,Home,5,No,Website
4,C0005,Company,Marleez,Co,1976-02-28,M,USA,California,Investment,5,No,Website


In [4]:
# Feature engineering: Calculate customer age
clients["date_of_birth"] = pd.to_datetime(
    clients["date_of_birth"],
    errors="coerce"
)

# Keep the reference date consistent with the earlier EDA
reference_date = pd.Timestamp("2025-01-01")

birth = clients["date_of_birth"]

clients["age"] = (
    reference_date.year - birth.dt.year
    - (
        (birth.dt.month > reference_date.month)
        | (
            (birth.dt.month == reference_date.month)
            & (birth.dt.day > reference_date.day)
        )
    ).astype(int)
)

# Flag impossible or implausible ages for investigation
invalid_age = (
    clients["age"].notna()
    & ~clients["age"].between(18, 110)
)

print("Missing ages:", clients["age"].isna().sum())
print("Ages requiring review:", invalid_age.sum())

display(clients[["date_of_birth", "age"]].head())

Missing ages: 0
Ages requiring review: 0


,date_of_birth,age
0,1968-05-11,56
1,1962-11-26,62
2,1959-04-07,65
3,1959-11-25,65
4,1976-02-28,48


In [5]:
# Select features for clustering
categorical_features = [
    "client_type",
    "gender",
    "country",
    "region",
    "acquisition_purpose",
    "loan_applied",
    "referral_channel"
]

numeric_features = [
    "age",
    "satisfaction_score"
]

feature_columns = categorical_features + numeric_features

# Check that every required column exists
missing_columns = [
    col for col in feature_columns
    if col not in clients.columns
]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

X = clients[feature_columns].copy()

print("Selected features:", X.shape)
display(X.head())

Selected features: (2000, 9)


,client_type,gender,country,region,acquisition_purpose,loan_applied,referral_channel,age,satisfaction_score
0,Individual,F,USA,California,Home,Yes,Website,56,4
1,Individual,M,USA,California,Home,No,Website,62,1
2,Individual,M,USA,California,Home,Yes,Agency,65,4
3,Individual,M,USA,California,Home,No,Website,65,5
4,Company,M,USA,California,Investment,No,Website,48,5


In [6]:
# Check feature values
print("Missing values by selected feature:")
display(X.isnull().sum())

print("\nCategorical values:")
for col in categorical_features:
    print(f"\n{col}")
    print(X[col].value_counts(dropna=False))

print("\nNumeric summary:")
display(X[numeric_features].describe())

Missing values by selected feature:


,0
client_type,0
gender,0
country,0
region,0
acquisition_purpose,0
loan_applied,0
referral_channel,0
age,0
satisfaction_score,0



Categorical values:

client_type
client_type
Individual    1897
Company        103
Name: count, dtype: int64

gender
gender
M    1012
F     988
Name: count, dtype: int64

country
country
USA          1538
UK             95
Canada         85
Germany        56
France         53
Belgium        43
Mexico         40
Australia      39
Russia         36
Denmark        15
Name: count, dtype: int64

region
region
California                633
Nevada                    143
Colorado                  118
Arizona                   108
Oregon                     96
Utah                       87
Washington                 73
Virginia                   65
Texas                      56
Florida                    50
New York                   49
Georgia                    34
England                    29
Alberta                    27
Scotland                   25
Ohio                       24
Northern Ireland           24
British Columbia           20
Wales                      17
Brittany             

,age,satisfaction_score
count,2000.000000,2000.000000
mean,54.100500,3.029000
std,17.343581,1.413562
min,24.000000,1.000000
25%,39.000000,2.000000
50%,55.000000,3.000000
75%,69.000000,4.000000
max,93.000000,5.000000


In [7]:
# Encode categorical features and scale numeric features
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, categorical_features),
    ("numeric", numeric_pipeline, numeric_features)
])

X_processed = preprocessor.fit_transform(X)

# Convert sparse matrix to dense only for this small dataset
if hasattr(X_processed, "toarray"):
    X_processed = X_processed.toarray()

feature_names = preprocessor.get_feature_names_out()

X_processed_df = pd.DataFrame(
    X_processed,
    columns=feature_names,
    index=clients.index
)

print("Original selected features:", X.shape)
print("Processed feature matrix:", X_processed_df.shape)

display(X_processed_df.head())

Original selected features: (2000, 9)
Processed feature matrix: (2000, 80)


,categorical__client_type_Company,categorical__client_type_Individual,categorical__gender_F,categorical__gender_M,categorical__country_Australia,categorical__country_Belgium,categorical__country_Canada,categorical__country_Denmark,categorical__country_France,categorical__country_Germany,...,categorical__region_Zealand,categorical__acquisition_purpose_Home,categorical__acquisition_purpose_Investment,categorical__loan_applied_No,categorical__loan_applied_Yes,categorical__referral_channel_Agency,categorical__referral_channel_Client,categorical__referral_channel_Website,numeric__age,numeric__satisfaction_score
0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.109549,0.687089
1,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.455585,-1.435740
2,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.628603,0.687089
3,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.628603,1.394698
4,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,-0.351832,1.394698


In [8]:
# Verify scaling
scaled_columns = [
    col for col in X_processed_df.columns
    if col.startswith("numeric__")
]

print("Scaled numerical feature summary:")
display(X_processed_df[scaled_columns].describe().T)

Scaled numerical feature summary:


,count,mean,std,min,25%,50%,75%,max
numeric__age,2000.0,1.882938e-16,1.00025,-1.735975,-0.870886,0.051877,0.859293,2.243437
numeric__satisfaction_score,2000.0,1.065814e-17,1.00025,-1.435740,-0.728130,-0.020521,0.687089,1.394698


In [9]:
# Save your processed data
# Save the encoded and scaled feature matrix
X_processed_df.to_csv(
    "clients_features_processed.csv",
    index=False
)

# Save the fitted preprocessing pipeline for later reuse
joblib.dump(
    preprocessor,
    "buyer_segmentation_preprocessor.joblib"
)

# Save client IDs separately to map cluster assignments back later
clients[["client_id"]].to_csv(
    "client_ids_for_clustering.csv",
    index=False
)

print("Feature preprocessing completed.")
print("Saved: clients_features_processed.csv")
print("Saved: buyer_segmentation_preprocessor.joblib")
print("Saved: client_ids_for_clustering.csv")

Feature preprocessing completed.
Saved: clients_features_processed.csv
Saved: buyer_segmentation_preprocessor.joblib
Saved: client_ids_for_clustering.csv
